# Exercise: An AI-Powered SQL Agent

Build an agent that answers questions about a real dataset by writing and running SQL, using the native chat-model interface and the hand-written agentic loop from notebook 1. The data is the January 2026 NYC yellow taxi trips, queried through DuckDB.

Fill in each `# YOUR CODE HERE`. A fully worked solution is in [`solutions/2_exercise_sql_agent.ipynb`](solutions/2_exercise_sql_agent.ipynb).

## Learning Objectives

At the end of this exercise, you should be able to:

- Build a SQL agent from scratch that inspects a schema and runs queries through tools.
- Test an agent by asserting on its answer and on the order of its tool calls.
- Evaluate open-ended answers with a second model acting as a judge.
- Track the token usage and estimated cost of a run.

## Setup

We use DuckDB and the shared chat-model adapter. Select the provider and its API key in `.env`. This cell is given.

In [ ]:
import json
import os
import urllib.request
from collections.abc import Callable

import duckdb
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage

from llm_config import create_chat_model

In [ ]:
load_dotenv()

model = create_chat_model()

## Load the data (given)

We download one month of NYC yellow taxi trips as a Parquet file (about 64 MB) and load it into a DuckDB table named `trips`. This cell is given.

In [ ]:
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
)
PARQUET_FILE = "yellow_tripdata_2026-01.parquet"

if not os.path.exists(PARQUET_FILE):
    print("Downloading the taxi data (about 64 MB) ...")
    urllib.request.urlretrieve(DATA_URL, PARQUET_FILE)

con = duckdb.connect("taxi.db")
con.execute(f"CREATE TABLE IF NOT EXISTS trips AS SELECT * FROM '{PARQUET_FILE}'")
row_count_result = con.execute("SELECT COUNT(*) FROM trips").fetchone()
if row_count_result is None:
    raise RuntimeError("The row-count query returned no result.")
row_count = row_count_result[0]
row_count

## Task 1: the tools

Implement the two tools the agent will use. `get_schema` returns the columns and types of the `trips` table, and `run_sql` runs a query and returns up to 50 result rows. The agent cannot touch the database itself: it can only ask us to call these functions.

In [ ]:
def get_schema() -> str:
    """Return the columns and types of the trips table."""
    # YOUR CODE HERE
    # Hint: run "DESCRIBE trips" with con.execute(...).fetchall(), then format
    # each row as "name: type".
    raise NotImplementedError("Complete Task 1: implement get_schema().")


def run_sql(query: str) -> str:
    """Run a SQL query against the trips table and return up to 50 result rows."""
    # YOUR CODE HERE
    # Hint: str(con.sql(query).limit(50)) renders a readable table.
    raise NotImplementedError("Complete Task 1: implement run_sql().")


def parse_tool_arguments(
    raw_arguments: str | dict[str, object] | None,
) -> dict[str, object]:
    """Parse a tool-call JSON object and reject other JSON values."""
    arguments = (
        json.loads(raw_arguments or "{}")
        if isinstance(raw_arguments, str)
        else (raw_arguments or {})
    )
    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a JSON object.")
    return arguments


tool_functions: dict[str, Callable[..., object]] = {
    "get_schema": get_schema,
    "run_sql": run_sql,
}

## Task 2: describe the tools to the model

Describe both tools to the model as JSON schemas, the same shape you saw in notebook 1. `get_schema` takes no arguments; `run_sql` takes a single `query` string.

In [ ]:
tools: list[dict[str, object]] = [
    # YOUR CODE HERE
]

## Task 3: the agent loop

Build the agentic loop, reusing the pattern from notebook 1. The system prompt is given. Limit the loop to eight model calls so a repeated tool request cannot run indefinitely. In the loop, add up the token usage the API reports (`response.usage.prompt_tokens` and `response.usage.completion_tokens`) and record the name of every tool the model calls, so the tests below can inspect them.

In [ ]:
SYSTEM_PROMPT = (
    "You are a SQL analyst for a DuckDB table named trips. "
    "Always call get_schema first to inspect the columns, then write and run one "
    "SQL query with run_sql, and finally answer the question using the query result. "
    "Be concise."
)


def run_sql_agent(question: str) -> dict:
    messages: list[BaseMessage | dict[str, object]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tool_calls_made = []
    prompt_tokens = completion_tokens = 0

    # YOUR CODE HERE
    # Loop, as in notebook 1:
    #  - call model.bind_tools(tools).invoke(messages)
    #  - add response.usage_metadata["input_tokens"] / ["output_tokens"]
    #    to the counters
    #  - if the reply has no tool_calls, append the final answer and stop
    #  - otherwise append the AIMessage, validate call["args"], run each
    #    call["name"], then append a {"role": "tool", "tool_call_id": call["id"],
    #    "content": ...} message and record the tool name

    return {
        "answer": messages[-1]["content"],
        "tool_calls": tool_calls_made,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }

## Ask the agent a question

Once the loop works, this should answer the question by inspecting the schema and writing its own SQL.

In [ ]:
result = run_sql_agent("How many trips had more than 5 passengers?")
result["answer"]

## Task 4: assert the answer is correct

Compute the true value directly from the database, then assert that the agent's answer contains it. This catches wrong SQL and wrong arithmetic.

In [ ]:
expected_result = con.execute(
    "SELECT COUNT(*) FROM trips WHERE passenger_count > 5"
).fetchone()
if expected_result is None:
    raise RuntimeError("The validation query returned no result.")
expected = expected_result[0]

# YOUR CODE HERE
# Assert that the expected count appears in result["answer"], then print a pass message.

## Task 5: assert the tool-call order

A correct answer is not enough. The agent should inspect the schema before it writes SQL, so it does not guess column names. Assert that the first tool call was `get_schema` and that `run_sql` was used.

In [ ]:
# YOUR CODE HERE
# assert result["tool_calls"][0] == "get_schema" and "run_sql" in result["tool_calls"]

## Task 6: an LLM as a judge

Some answers cannot be checked with an exact match, for example a one-sentence explanation. Write a `judge` function that asks a second model to grade an answer against a natural-language criterion and reply with only `PASS` or `FAIL`.

In [ ]:
def judge(question: str, answer: str, criteria: str) -> str:
    """Ask a second model whether the answer meets a natural-language criterion."""
    # YOUR CODE HERE
    # Send a chat completion whose system prompt tells the grader to reply only
    # PASS or FAIL, and whose user message contains the question, answer, and
    # criteria. Return the reply text.
    raise NotImplementedError("Complete Task 5: implement judge().")

## Task 7: several scenarios

Real test suites cover more than one case. Add a few scenarios, each pairing a question with the criterion its answer must meet, then run the agent on each and let the judge grade it.

In [ ]:
scenarios = [
    # YOUR CODE HERE: add {"question": ..., "criteria": ...} dicts,
    # for example the average trip distance or the total number of trips.
]

for case in scenarios:
    run = run_sql_agent(case["question"])
    verdict = judge(case["question"], run["answer"], case["criteria"])
    print(f"[{verdict}] {case['question']}")
    print("   ->", run["answer"])
    print("   tools:", run["tool_calls"])

## Task 8: track the cost

Every response reports how many tokens it used, which you summed in the loop. Groq bills per token, so turn the counts into an estimated cost. The rates below are approximate and change over time.

In [ ]:
# Illustrative Groq rates, not a bill; replace both for the selected model.
INPUT_RATE = 0.075 / 1_000_000  # USD per input token (approximate)
OUTPUT_RATE = 0.30 / 1_000_000  # USD per output token (approximate)

# YOUR CODE HERE
# Combine result["prompt_tokens"] and result["completion_tokens"] with the rates
# into an estimated cost, and print the token counts and the estimate.

## Summary

When you are done, your notebook should:

- Build a SQL agent from scratch that inspects the schema and runs queries through the two tools.
- Assert on both the agent's answer and the order of its tool calls.
- Use a second model as a judge to grade an answer against a natural-language criterion.
- Track token usage and estimate the cost of a run.

The agent is the same hand-written loop from notebook 1, pointed at real tools over a real database.

## References & Further Reading

- [**Groq Tool Use**](https://console.groq.com/docs/tool-use): Function calling and the tool-call loop.
- [**DuckDB Documentation**](https://duckdb.org/docs/): Querying Parquet and running SQL in-process.
- [**NYC TLC Trip Record Data**](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): The source of the taxi dataset.
- [**Groq Console**](https://console.groq.com/playground): Create a free API key and try the model used here.